# Airbnb Price Prediction — Full ML Pipeline
This notebook covers data preprocessing, model training, evaluation, and MLflow experiment tracking.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

from src.dataloader import load_data
from src.preprocessor import preprocess, split_features_target

print('All imports successful!')

## Step 1 — Load Data

In [ ]:
df_raw = load_data()
print('Raw shape:', df_raw.shape)
df_raw.head()

## Step 2 — Preprocess Data

In [ ]:
df_clean = preprocess(df_raw)
print('Clean shape:', df_clean.shape)
df_clean.head()

## Step 3 — Split Features and Target

In [ ]:
X, y = split_features_target(df_clean, target_col='price')
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

## Step 4 — Train Models with MLflow Tracking

In [ ]:
def compute_metrics(y_true, y_pred):
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_true, y_pred)
    return {'mse': round(mse, 4), 'rmse': round(rmse, 4), 'r2': round(r2, 4)}

mlflow.set_experiment('Airbnb_Pricing')
results = {}
print('MLflow experiment set: Airbnb_Pricing')

In [ ]:
# ── Linear Regression ──
with mlflow.start_run(run_name='Linear_Regression'):
    model = LinearRegression()
    model.fit(X_train, y_train)
    metrics = compute_metrics(y_test, model.predict(X_test))
    mlflow.log_params({'model_type': 'LinearRegression', 'test_size': 0.2})
    mlflow.log_metrics(metrics)
    mlflow.sklearn.log_model(model, 'model')
    results['Linear_Regression'] = {'metrics': metrics, 'run_id': mlflow.active_run().info.run_id}
print('Linear Regression:', results['Linear_Regression']['metrics'])

In [ ]:
# ── Ridge Regression ──
with mlflow.start_run(run_name='Ridge_Regression'):
    model = Ridge(alpha=1.0)
    model.fit(X_train, y_train)
    metrics = compute_metrics(y_test, model.predict(X_test))
    mlflow.log_params({'model_type': 'Ridge', 'alpha': 1.0, 'test_size': 0.2})
    mlflow.log_metrics(metrics)
    mlflow.sklearn.log_model(model, 'model')
    results['Ridge_Regression'] = {'metrics': metrics, 'run_id': mlflow.active_run().info.run_id}
print('Ridge Regression:', results['Ridge_Regression']['metrics'])

In [ ]:
# ── Random Forest ──
with mlflow.start_run(run_name='Random_Forest'):
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    metrics = compute_metrics(y_test, model.predict(X_test))
    mlflow.log_params({'model_type': 'RandomForest', 'n_estimators': 100, 'random_state': 42})
    mlflow.log_metrics(metrics)
    mlflow.sklearn.log_model(model, 'model')
    results['Random_Forest'] = {'metrics': metrics, 'run_id': mlflow.active_run().info.run_id}
print('Random Forest:', results['Random_Forest']['metrics'])

## Step 5 — Compare Models

In [ ]:
summary = pd.DataFrame({name: info['metrics'] for name, info in results.items()}).T
print(summary.to_string())
summary

In [ ]:
# Bar chart comparison
summary[['mse', 'r2']].plot(kind='bar', figsize=(10, 5), colormap='Set2', edgecolor='black')
plt.title('Model Comparison — MSE and R²')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Step 6 — Register Best Model

In [ ]:
best_name   = summary['mse'].idxmin()
best_run_id = results[best_name]['run_id']
print(f'Best model: {best_name}')

registered = mlflow.register_model(
    model_uri=f'runs:/{best_run_id}/model',
    name='Best_Airbnb_Model_RandomForest'
)
print(f'Registered version: {registered.version}')